# Enhancing RAG with Contextual Retrieval
We will use an LLM to generate for each chunk and document a contextual sentence to improve its retrival accuracy and use in hybrid search.

- Load complex documents dataset
- Split the documents into chunks
- Generate the context sentence
- Enrich the chunk embedding vectors with the context


#### Visual Improvements
We will use rich library to make the output more readable, and supress warning messages.

In [1]:
from rich.console import Console
from rich_theme_manager import Theme, ThemeManager
import pathlib

theme_dir = pathlib.Path("themes")
theme_manager = ThemeManager(theme_dir=theme_dir)
dark = theme_manager.get("dark")

# Create a console with the dark theme
console = Console(theme=dark)

In [2]:
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')


## Loading a complex dataset of documents
We will load a complex dataset of scientific documents from Arxiv. Applying naive chunks on such documents will give poor results in RAG applications.

In [3]:
from datasets import load_dataset

dataset = load_dataset("jamescalam/ai-arxiv2", split="train")
console.print(dataset)

Dataset({
    features: ['id', 'title', 'summary', 'source', 'authors', 'categories', 'comment', 'journal_ref', 
'primary_category', 'published', 'updated', 'content', 'references'],
    num_rows: 2673
})

## Split the documents into Chunks
We will use the statistical chunker that we used in a previous notebook.

In [4]:
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
import os
from semantic_router.encoders import OpenAIEncoder

encoder = OpenAIEncoder(name="text-embedding-3-small")

In [6]:
from semantic_chunkers import StatisticalChunker
import logging

logging.disable(logging.CRITICAL)

chunker = StatisticalChunker(
    encoder=encoder,
    min_split_tokens=100,
    max_split_tokens=500,
)

In [7]:
chunks_0 = chunker(docs=[dataset["content"][0]])

  0%|          | 0/8 [00:00<?, ?it/s]

In [8]:
from rich.text import Text
from rich.panel import Panel

chunk_0_0 = ' '.join(chunks_0[0][0].splits)

content = Text(chunk_0_0)
console.print(Panel(content, title=f"Chunk 0", expand=False, border_style="bold"))

╭──────────────────────────────────────────────────── Chunk 0 ────────────────────────────────────────────────────╮
│ 4 2 0 2 n a J 8 ] G L . s c [ 1 v 8 8 0 4 0 . 1 0 4 2 : v i X r a # Mixtral of Experts Albert Q. Jiang,         │
│ Alexandre Sablayrolles, Antoine Roux, Arthur Mensch, Blanche Savary, Chris Bamford, Devendra Singh Chaplot,     │
│ Diego de las Casas, Emma Bou Hanna, Florian Bressand, Gianna Lengyel, Guillaume Bour, Guillaume Lample, LÃ©lio  │
│ Renard Lavaud, Lucile Saulnier, Marie-Anne Lachaux, Pierre Stock, Sandeep Subramanian, Sophia Yang, Szymon      │
│ Antoniak, Teven Le Scao, ThÃ©ophile Gervet, Thibaut Lavril, Thomas Wang, TimothÃ©e Lacroix, William El Sayed    │
│ Abstract We introduce Mixtral 8x7B, a Sparse Mixture of Experts (SMoE) language model. Mixtral has the same     │
│ architecture as Mistral 7B, with the difference that each layer is composed of 8 feedforward blocks (i.e.       │
│ experts). For every token, at each layer, a router network selects two experts to process the current state and │
│ combine their outputs. Even though each token only sees two experts, the selected experts can be different at   │
│ each timestep. As a result, each token has access to 47B parameters, but only uses 13B active parameters during │
│ inference. Mixtral was trained with a context size of 32k tokens and it outperforms or matches Llama 2 70B and  │
│ GPT-3.5 across all evaluated benchmarks. In particular, Mixtral vastly outperforms Llama 2 70B on mathematics,  │
│ code generation, and multilingual benchmarks. We also provide a model fine- tuned to follow instructions,       │
│ Mixtral 8x7B â Instruct, that surpasses GPT-3.5 Turbo, Claude-2.1, Gemini Pro, and Llama 2 70B â                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Generate the context sentence
We will use Anthropic Claude for the generation of the context. It is one of the best summarization LLM, and it introduced the Prompt Caching that is great for the generation of the context for many chunks of the same document.

In [12]:
# import anthropic
# client = anthropic.Anthropic()

from openai import OpenAI

client = OpenAI()


In [15]:
DOCUMENT_CONTEXT_PROMPT = """
<document>
{doc_content}
</document>
"""

CHUNK_CONTEXT_PROMPT = """
Here is the chunk we want to situate within the whole document
<chunk>
{chunk_content}
</chunk>

Please give a short succinct context to situate this chunk within the overall document for the purposes of improving search retrieval of the chunk.
Answer only with the succinct context and nothing else.
"""

def situate_context(doc: str, chunk: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=1024,
        temperature=0.0,
        messages=[
            {
                "role": "user",
                "content": (
                    DOCUMENT_CONTEXT_PROMPT.format(doc_content=doc)
                    + "\n"
                    + CHUNK_CONTEXT_PROMPT.format(chunk_content=chunk)
                ),
            }
        ],
    )
    return response.choices[0].message.content


In [16]:
chunk_context = situate_context(dataset["content"][0], chunk_0_0)
console.print(chunk_context)

The chunk is an excerpt from the abstract of the document, which introduces the Mixtral 8x7B model, a Sparse 
Mixture of Experts (SMoE) language model that outperforms existing models like Llama 2 70B and GPT-3.5 across 
various benchmarks, particularly in mathematics, code generation, and multilingual tasks.

In [17]:
chunk_0_5 = ' '.join(chunks_0[0][5].splits)

In [18]:
second_chunk_context = situate_context(dataset["content"][0], chunk_0_5)
console.print(second_chunk_context)

The chunk is part of Section 2, titled "Architectural details," which provides an overview of the Mixtral 8x7B 
model's architecture, specifically focusing on the Sparse Mixture of Experts (MoE) layer. It includes key 
parameters of the model and introduces the mechanism by which the MoE layer operates, detailing how the output is 
computed based on the gating network's selection of expert networks.

## Enrich the chunk embedding vectors with the context
### Concatenate the generated context to the chunk text
We will iterate over all the chunks. This can take some time based on the number of chunks.

In [20]:
arxiv_id = dataset[0]["id"]
refs = list(dataset[0]["references"].values())
doc_text = dataset[0]["content"]
title = dataset[0]["title"]

from tqdm import tqdm

corpus_json = []
for i, chunk in tqdm(enumerate(chunks_0[0]), total=len(chunks_0[0]), desc="Processing chunks"):
    chunk_text = ' '.join(chunk.splits)
    contextualized_text = situate_context(doc_text, chunk_text)
    corpus_json.append({
        "id": i,
        "text": f"{chunk_text}\n\n{contextualized_text}",
        "metadata" : {
            "title": title,
            "arxiv_id": arxiv_id,
            "references": refs
        }
    })

Processing chunks: 100%|██████████| 49/49 [01:47<00:00,  2.19s/it]


In [21]:
console.print(corpus_json[:2])

[
    {
        'id': 0,
        'text': "4 2 0 2 n a J 8 ] G L . s c [ 1 v 8 8 0 4 0 . 1 0 4 2 : v i X r a # Mixtral of Experts Albert Q. 
Jiang, Alexandre Sablayrolles, Antoine Roux, Arthur Mensch, Blanche Savary, Chris Bamford, Devendra Singh Chaplot, 
Diego de las Casas, Emma Bou Hanna, Florian Bressand, Gianna Lengyel, Guillaume Bour, Guillaume Lample, LÃ©lio 
Renard Lavaud, Lucile Saulnier, Marie-Anne Lachaux, Pierre Stock, Sandeep Subramanian, Sophia Yang, Szymon 
Antoniak, Teven Le Scao, ThÃ©ophile Gervet, Thibaut Lavril, Thomas Wang, TimothÃ©e Lacroix, William El Sayed 
Abstract We introduce Mixtral 8x7B, a Sparse Mixture of Experts (SMoE) language model. Mixtral has the same 
architecture as Mistral 7B, with the difference that each layer is composed of 8 feedforward blocks (i.e. experts).
For every token, at each layer, a router network selects two experts to process the current state and combine their
outputs. Even though each token only sees two experts, the selected experts can be different at each timestep. As a
result, each token has access to 47B parameters, but only uses 13B active parameters during inference. Mixtral was 
trained with a context size of 32k tokens and it outperforms or matches Llama 2 70B and GPT-3.5 across all 
evaluated benchmarks. In particular, Mixtral vastly outperforms Llama 2 70B on mathematics, code generation, and 
multilingual benchmarks. We also provide a model fine- tuned to follow instructions, Mixtral 8x7B â Instruct, that 
surpasses GPT-3.5 Turbo, Claude-2.1, Gemini Pro, and Llama 2 70B â\n\nThe chunk is an excerpt from the abstract of 
the document, which introduces the Mixtral 8x7B model, a Sparse Mixture of Experts (SMoE) language model that 
builds on the architecture of Mistral 7B. It highlights the model's unique features, performance benchmarks against
other models like Llama 2 70B and GPT-3.5, and its capabilities in mathematics, code generation, and multilingual 
tasks.",
        'metadata': {'title': 'Mixtral of Experts', 'arxiv_id': '2401.04088', 'references': ['1905.07830']}
    },
    {
        'id': 1,
        'text': "chat model on human bench- marks. Both the base and instruct models are released under the Apache 
2.0 license. Code: https://github.com/mistralai/mistral-src Webpage: https://mistral.ai/news/mixtral-of-experts/ # 
Introduction In this paper, we present Mixtral 8x7B, a sparse mixture of experts model (SMoE) with open weights, 
licensed under Apache 2.0. Mixtral outperforms Llama 2 70B and GPT-3.5 on most benchmarks. As it only uses a subset
of its parameters for every token, Mixtral allows faster inference speed at low batch-sizes, and higher throughput 
at large batch-sizes. Mixtral is a sparse mixture-of-experts network. It is a decoder-only model where the 
feedforward block picks from a set of 8 distinct groups of parameters. At every layer, for every token, a router 
network chooses two of these groups (the â\n\nThe chunk is part of the introduction section of the document, which 
presents Mixtral 8x7B, a Sparse Mixture of Experts (SMoE) language model. It highlights the model's architecture, 
performance advantages over Llama 2 70B and GPT-3.5, and its efficient use of parameters for improved inference 
speed and throughput.",
        'metadata': {'title': 'Mixtral of Experts', 'arxiv_id': '2401.04088', 'references': ['1905.07830']}
    }
]

### Saving the corpus_json in a file
We will want to use it in the next notebook.

In [22]:
import json

with open('data/corpus.json', 'w') as f:
    json.dump(corpus_json, f)